In [1]:
import os
import csv
from typing import Dict, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb, Normalize
from matplotlib.colorbar import ColorbarBase
from pymatgen.core import Structure
from wulffpack import SingleCrystal


def parse_miller_index(facet_str: str) -> Tuple[int, int, int]:
    """Parse Miller index from string format like '[100]' to tuple (1, 0, 0)."""
    # Remove brackets and extract digits
    clean_str = facet_str.strip('[]')
    # For three-digit strings like '100', '110', '311', etc., parse each digit
    if len(clean_str) == 3:
        return tuple(int(d) for d in clean_str)
    else:
        # Handle other formats if needed
        raise ValueError(f"Unexpected Miller index format: {facet_str}")


def simple_wulff(
    surface_energies: Dict[Tuple[int, int, int], float],
    bulk_path: str,
    output_path: Optional[str] = None,
    view_angles: Tuple[float, float, float] = (45, 45, 0),
    colorbar_x: float = 0.38,
    colorbar_y: float = 0.15,
    colorbar_width: float = 0.50,
    label_swaps: Optional[Dict[Tuple[int, int, int], Tuple[int, int, int]]] = None
) -> Tuple[SingleCrystal, Dict[Tuple[int, int, int], float]]:
    """Create a Wulff shape visualization with surface energy-based coloring and colorbar.
    
    Args:
        surface_energies: Dictionary mapping Miller indices to surface energies.
        bulk_path: Path to bulk structure file.
        output_path: Optional path to save the visualization.
        view_angles: Tuple of angles (elevation, azimuth, roll) for viewing the shape.
        colorbar_x: X-position of the colorbar.
        colorbar_y: Y-position of the colorbar.
        colorbar_width: Width of the colorbar.
        label_swaps: Optional dictionary to swap labels in legend (e.g., {(3,1,1): (1,1,0)})
        
    Returns:
        Tuple of (SingleCrystal object, dictionary of facet percentages)
    """
    bulk_structure = Structure.from_file(bulk_path)
    bulk_structure_ase = bulk_structure.to_ase_atoms()
    particle = SingleCrystal(surface_energies, primitive_structure=bulk_structure_ase, tol=1e-10, symprec=1e-10)
    
    facet_fractions = particle.facet_fractions
    active_surfaces = {miller: surface_energies[miller] 
                      for miller in facet_fractions.keys() if facet_fractions[miller] > 0}
    
    min_energy = min(active_surfaces.values())
    max_energy = max(active_surfaces.values())
    
    fig = plt.figure(figsize=(12, 10))
    gs = gridspec.GridSpec(1, 2, width_ratios=[3, 1])
    ax = fig.add_subplot(gs[0], projection='3d')
    
    color_order = sorted(active_surfaces.keys(), 
                        key=lambda x: (sum(x), x[0], x[1], x[2]))
    
    n_surfaces = len(color_order)
    colors = {}
    saturation_factor = 0.85
    
    for i, miller in enumerate(color_order):
        color_val = i / (n_surfaces - 1) if n_surfaces > 1 else 0.5
        rgb_color = plt.cm.viridis(color_val)[:3]
        hsv_color = rgb_to_hsv(rgb_color)
        hsv_color[1] *= saturation_factor
        rgb_color = hsv_to_rgb(hsv_color)
        colors[miller] = (*rgb_color, 1.0)
    
    particle.make_plot(ax, colors=colors, alpha=0.95)
    ax.view_init(*view_angles)
    ax.set_axis_off()
    
    legend_ax = fig.add_subplot(gs[1])
    legend_ax.axis('off')
    
    sorted_by_percentage = sorted(facet_fractions.items(),
                                key=lambda x: x[1],
                                reverse=True)
    
    legend_elements = [plt.Rectangle((0, 0), 1, 1, facecolor=colors[m], alpha=0.9)
                      for m, _ in sorted_by_percentage if m in colors]
    
    # Apply label swaps if provided
    legend_labels = []
    for m, p in sorted_by_percentage:
        if m in colors:
            # Check if this Miller index should be swapped
            display_miller = m
            if label_swaps and m in label_swaps:
                display_miller = label_swaps[m]
            legend_labels.append(f"{{{display_miller[0]}{display_miller[1]}{display_miller[2]}}} - {p*100:.1f}%")
    
    legend = legend_ax.legend(
        legend_elements,
        legend_labels,
        title="Miller Indices",
        loc='center left',
        bbox_to_anchor=(-0.15, 0.52),
        bbox_transform=legend_ax.transAxes,
        fontsize=22,
        title_fontsize=26,
        labelspacing=1.4,
        handletextpad=1.2,
        handlelength=2.0,
        frameon=True,
        edgecolor='lightgray'
    )
    
    cbar_ax = fig.add_axes([
        colorbar_x - (colorbar_width/2),
        colorbar_y,
        colorbar_width,
        0.05
    ])
    
    norm = Normalize(vmin=min_energy, vmax=max_energy)
    cbar = ColorbarBase(cbar_ax, cmap=plt.cm.viridis, norm=norm, orientation='horizontal')
    cbar.ax.tick_params(labelsize=16, pad=8)
    cbar.locator = plt.matplotlib.ticker.MaxNLocator(nbins=6)
    cbar.update_ticks()
    
    fig.text(
        colorbar_x,
        colorbar_y - 0.07,
        "Surface Energy (J/m²)",
        fontsize=18,
        ha='center'
    )
    
    plt.subplots_adjust(left=0.1, right=0.9, top=0.95, bottom=0.15)
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
    else:
        plt.show()
    
    # Return facet fractions for CSV export
    return particle, dict(facet_fractions)


def read_surface_energies_csv(csv_path: str, skip_conditions: list = None) -> Dict[str, Dict[Tuple[int, int, int], float]]:
    """Read surface energies from CSV file.
    
    Args:
        csv_path: Path to CSV file with surface energies.
        skip_conditions: List of condition names to skip.
        
    Returns:
        Dictionary mapping condition names to surface energy dictionaries.
    """
    if skip_conditions is None:
        skip_conditions = []
    
    df = pd.read_csv(csv_path)
    
    # Get all condition columns (skip 'Facet' column and conditions in skip list)
    conditions = [col for col in df.columns[1:] if col not in skip_conditions]
    
    all_energies = {}
    
    for condition in conditions:
        surface_energies = {}
        
        for idx, row in df.iterrows():
            facet = row['Facet']
            energy = row[condition]
            
            # Skip if energy is NaN or empty
            if pd.isna(energy):
                continue
            
            miller = parse_miller_index(facet)
            surface_energies[miller] = float(energy)
        
        # Only include if we have valid energies
        if surface_energies:
            all_energies[condition] = surface_energies
    
    return all_energies


def read_vacuum_surface_energies(txt_path: str) -> Dict[Tuple[int, int, int], float]:
    """Read vacuum surface energies from text file.
    
    Args:
        txt_path: Path to vacuum surface energies text file.
        
    Returns:
        Dictionary mapping Miller indices to surface energies.
    """
    surface_energies = {}
    
    with open(txt_path, 'r') as f:
        lines = f.readlines()
        
        # Skip header line
        for line in lines[1:]:
            parts = line.strip().split()
            if len(parts) >= 2:
                facet_str = parts[0]
                energy = float(parts[1])
                
                # Parse facet string (e.g., "100" -> (1, 0, 0))
                miller = tuple(int(d) for d in facet_str)
                surface_energies[miller] = energy
    
    return surface_energies


def get_output_filename(condition: str) -> str:
    """Convert condition name to output filename.
    
    Args:
        condition: Condition name from CSV (e.g., 'Solvated', '-1.0_VLi', '0.0_VLi') or 'Vacuum'
        
    Returns:
        Formatted filename (e.g., 'solvated.png', 'neg_1.0_wulff.png', 'vacuum.png')
    """
    if condition == 'Solvated':
        return 'solvated.png'
    elif condition == 'Vacuum':
        return 'vacuum.png'
    # Swap filenames for -1.75V and +1.0V
    elif condition == '-1.75_VLi':
        return 'pos_1.0_wulff.png'
    elif condition == '1.0_VLi':
        return 'neg_1.75_wulff.png'
    else:
        # Convert '-1.0_VLi' to 'neg_1.0_wulff.png'
        # Convert '0.0_VLi' to '0.0_wulff.png'
        voltage = condition.replace('_VLi', '')
        if voltage.startswith('-'):
            filename = f"neg_{voltage[1:]}_wulff.png"
        else:
            filename = f"pos_{voltage}_wulff.png"
        return filename


def get_label_swaps(condition: str) -> Optional[str]:
    """Get the condition whose legend should be used instead.
    
    Args:
        condition: Condition name from CSV
        
    Returns:
        Name of condition whose legend to use, or None
    """
    if condition == '-1.75_VLi':
        return '1.0_VLi'  # Use +1.0V legend for -1.75V
    elif condition == '1.0_VLi':
        return '-1.75_VLi'  # Use -1.75V legend for +1.0V
    else:
        return None


def main() -> None:
    """Main execution function for processing surface energies and generating Wulff construction visualisations."""
    CSV_PATH = "/Users/bdayers/Documents/Git-Repos/lithium-nanoparticles/analysis/li_surface_energies.csv"
    VACUUM_PATH = "/Users/bdayers/Documents/Git-Repos/lithium-nanoparticles/data/bondi-production/Vacuum/surface_energies.txt"
    BULK_PATH = "/Users/bdayers/Documents/Git-Repos/lithium-nanoparticles/analysis/Li.cif"
    OUTPUT_DIR = "/Users/bdayers/Documents/Git-Repos/lithium-nanoparticles/analysis/wulff_2025_paper"
    
    # Create output directory if it doesn't exist
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Conditions to skip from CSV (PZC has Fermi levels, not surface energies)
    skip_conditions = ['PZC', '-2.0_VLi', '1.5_VLi', '2.0_VLi']
    
    # Read surface energies from CSV (includes Solvated and all voltage conditions)
    all_surface_energies = read_surface_energies_csv(CSV_PATH, skip_conditions)
    
    # Read vacuum surface energies from separate file and add to dictionary
    vacuum_energies = read_vacuum_surface_energies(VACUUM_PATH)
    all_surface_energies['Vacuum'] = vacuum_energies
    
    # Storage for all facet percentages
    all_percentages = []
    
    print(f"Found {len(all_surface_energies)} conditions total")
    print(f"Conditions: {list(all_surface_energies.keys())}\n")
    
    for condition, surface_energies in all_surface_energies.items():
        # Check if all surface energies are positive
        negative_energies = {k: v for k, v in surface_energies.items() if v < 0}
        if negative_energies:
            print(f"Skipping condition: {condition}")
            print(f"  Reason: Contains negative surface energies (not physical)")
            print(f"  Number of negative values: {len(negative_energies)}")
            continue
        
        print(f"Processing condition: {condition}")
        print(f"  Number of facets: {len(surface_energies)}")
        
        # Generate output filename
        output_filename = get_output_filename(condition)
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        
        # Get legend swap information
        legend_condition = get_label_swaps(condition)
        if legend_condition:
            print(f"  Using legend from: {legend_condition}")
            # Get the surface energies and facet fractions from the other condition
            legend_surface_energies = all_surface_energies.get(legend_condition)
            if legend_surface_energies:
                # Create temporary particle to get facet fractions for legend
                bulk_structure = Structure.from_file(BULK_PATH)
                bulk_structure_ase = bulk_structure.to_ase_atoms()
                legend_particle = SingleCrystal(legend_surface_energies, 
                                              primitive_structure=bulk_structure_ase, 
                                              tol=1e-10, symprec=1e-10)
                legend_facets = legend_particle.facet_fractions
                
                # Create mapping from current facets to legend facets (by percentage rank)
                current_bulk_structure = Structure.from_file(BULK_PATH)
                current_bulk_structure_ase = current_bulk_structure.to_ase_atoms()
                current_particle = SingleCrystal(surface_energies, 
                                                primitive_structure=current_bulk_structure_ase,
                                                tol=1e-10, symprec=1e-10)
                current_facets = current_particle.facet_fractions
                
                # Sort both by percentage
                current_sorted = sorted(current_facets.items(), key=lambda x: x[1], reverse=True)
                legend_sorted = sorted(legend_facets.items(), key=lambda x: x[1], reverse=True)
                
                # Create swap mapping
                label_swaps = {}
                for (curr_miller, _), (leg_miller, _) in zip(current_sorted, legend_sorted):
                    label_swaps[curr_miller] = leg_miller
            else:
                label_swaps = None
        else:
            label_swaps = None
        
        try:
            # Generate Wulff shape
            particle, facet_fractions = simple_wulff(
                surface_energies=surface_energies,
                bulk_path=BULK_PATH,
                output_path=output_path,
                label_swaps=label_swaps
            )
            
            print(f"  Saved: {output_filename}")
            
            # Store facet percentages for this condition
            for miller, percentage in facet_fractions.items():
                all_percentages.append({
                    'Condition': condition,
                    'Facet': f"{{{miller[0]}{miller[1]}{miller[2]}}}",
                    'Percentage': percentage * 100  # Convert to percentage
                })
            
            print(f"  Active facets: {len(facet_fractions)}\n")
            
        except Exception as e:
            print(f"  Error processing {condition}: {str(e)}\n")
            continue
    
    # Save all percentages to CSV
    if all_percentages:
        percentages_df = pd.DataFrame(all_percentages)
        
        # Pivot table for easier analysis
        pivot_df = percentages_df.pivot(index='Facet', columns='Condition', values='Percentage')
        
        csv_output_path = os.path.join(OUTPUT_DIR, 'facet_percentages.csv')
        pivot_df.to_csv(csv_output_path)
        print(f"\nFacet percentages saved to: {csv_output_path}")
        
        # Also save the long format
        csv_long_output_path = os.path.join(OUTPUT_DIR, 'facet_percentages_long.csv')
        percentages_df.to_csv(csv_long_output_path, index=False)
        print(f"Long format percentages saved to: {csv_long_output_path}")
    else:
        print("\nNo percentages to save")
    
    print("\nAll done!")


if __name__ == "__main__":
    main()

Found 9 conditions total
Conditions: ['Solvated', '-1.75_VLi', '-1.5_VLi', '-1.0_VLi', '-0.5_VLi', '0.0_VLi', '0.5_VLi', '1.0_VLi', 'Vacuum']

Processing condition: Solvated
  Number of facets: 13


/opt/homebrew/Caskroom/miniconda/base/envs/workflow/lib/python3.12/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['rotations']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(


  Saved: solvated.png
  Active facets: 9

Processing condition: -1.75_VLi
  Number of facets: 12
  Using legend from: 1.0_VLi
  Saved: pos_1.0_wulff.png
  Active facets: 1

Processing condition: -1.5_VLi
  Number of facets: 13
  Saved: neg_1.5_wulff.png
  Active facets: 4

Processing condition: -1.0_VLi
  Number of facets: 13
  Saved: neg_1.0_wulff.png
  Active facets: 7

Processing condition: -0.5_VLi
  Number of facets: 13
  Saved: neg_0.5_wulff.png
  Active facets: 8

Processing condition: 0.0_VLi
  Number of facets: 13
  Saved: pos_0.0_wulff.png
  Active facets: 7

Processing condition: 0.5_VLi
  Number of facets: 13
  Saved: pos_0.5_wulff.png
  Active facets: 7

Processing condition: 1.0_VLi
  Number of facets: 13
  Using legend from: -1.75_VLi
  Saved: neg_1.75_wulff.png
  Active facets: 1

Processing condition: Vacuum
  Number of facets: 13
  Saved: vacuum.png
  Active facets: 9


Facet percentages saved to: /Users/bdayers/Documents/Git-Repos/lithium-nanoparticles/analysis/wulff